In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('../raw_data/features_enriched_data.csv',
                 parse_dates=['founded_at', 'first_funding_at', 'last_funding_at'])
print(f"Loaded: {df.shape}")

# Drop unknown status
df = df[df['status_enriched'] != 'unknown']

# Target: success (1) = acquired or operating, failure (0) = closed
df['target'] = (df['status_enriched'].isin(['acquired', 'operating'])).astype(int)

print(f"Rows after dropping unknown: {df.shape[0]}")
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\nClass balance: {df['target'].mean():.1%} success / {1-df['target'].mean():.1%} closed")

Loaded: (44363, 31)
Rows after dropping unknown: 43212

Target distribution:
target
1    32181
0    11031
Name: count, dtype: int64

Class balance: 74.5% success / 25.5% closed


In [7]:
# Engineered numeric features
engineered_num = [
    'avg_raised_per_round',
    'age_first_funding_days',
    'has_multiple_rounds',
    'funding_span_days',
    'avg_years_between_rounds',
]

# Raw numeric features
raw_num = [
    'funding_rounds',
    'funding_total_usd',
    'seed',
    'venture',
    'angel',
    'grant',
    'debt_financing',
    'private_equity',
    'round_A',
    'round_B',
    'round_C',
    'round_D',
    'round_E',
]

num_features = engineered_num + raw_num

# Categorical features — using engineered groups instead of raw high-cardinality columns
cat_features = ['region_group', 'industry_group']

X = df[num_features + cat_features].copy()
y = df['target'].copy()

print(f"Features: {len(num_features)} numeric + {len(cat_features)} categorical = {len(X.columns)} total")
print(f"\nNumeric: {num_features}")
print(f"Categorical: {cat_features}")
print(f"\nX shape: {X.shape}")

Features: 18 numeric + 2 categorical = 20 total

Numeric: ['avg_raised_per_round', 'age_first_funding_days', 'has_multiple_rounds', 'funding_span_days', 'avg_years_between_rounds', 'funding_rounds', 'funding_total_usd', 'seed', 'venture', 'angel', 'grant', 'debt_financing', 'private_equity', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E']
Categorical: ['region_group', 'industry_group']

X shape: (43212, 20)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} rows ({y_train.mean():.1%} success)")
print(f"Test:  {X_test.shape[0]} rows ({y_test.mean():.1%} success)")

Train: 34569 rows (74.5% success)
Test:  8643 rows (74.5% success)


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', RobustScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
])

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
!pip install xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, RocCurveDisplay)

models = {
    'LogReg (balanced)': Pipeline([
        ('prep', preprocessor),
        ('model', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
    ]),
    'Decision Tree (balanced)': Pipeline([
        ('prep', preprocessor),
        ('model', DecisionTreeClassifier(class_weight='balanced', random_state=42))
    ]),
    'Random Forest (balanced)': Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
        ))
    ]),
    'KNN (k=5)': Pipeline([
        ('prep', preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Gradient Boosting': Pipeline([
        ('prep', preprocessor),
        ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
    'XGBoost': Pipeline([
        ('prep', preprocessor),
        ('model', XGBClassifier(
            n_estimators=100, random_state=42, eval_metric='logloss',
            scale_pos_weight=(y_train == 1).sum() / (y_train == 0).sum()
        ))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, pipe in models.items():
    print(f"\nTraining: {name}...")
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    f1_closed = f1_score(y_test, y_pred, pos_label=0)
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')

    results.append({
        'Model': name,
        'AUC (test)': round(auc, 4),
        'F1 closed': round(f1_closed, 4),
        'CV AUC (mean)': round(cv_auc.mean(), 4),
        'CV AUC (std)': round(cv_auc.std(), 4),
    })
    print(f"  AUC: {auc:.4f} | F1 closed: {f1_closed:.4f} | CV AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")

  Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl (2.3 MB)

Training: LogReg (balanced)...
  AUC: 0.5715 | F1 closed: 0.3940 | CV AUC: 0.5754 ± 0.0053

Training: Decision Tree (balanced)...
  AUC: 0.5241 | F1 closed: 0.3053 | CV AUC: 0.5234 ± 0.0073

Training: Random Forest (balanced)...
  AUC: 0.5609 | F1 closed: 0.2293 | CV AUC: 0.5590 ± 0.0073

Training: KNN (k=5)...
  AUC: 0.5486 | F1 closed: 0.2313 | CV AUC: 0.5426 ± 0.0071

Training: Gradient Boosting...
  AUC: 0.6213 | F1 closed: 0.0108 | CV AUC: 0.6173 ± 0.0042

Training: XGBoost...
  AUC: 0.5930 | F1 closed: 0.0045 | CV AUC: 0.5947 ± 0.0049


In [12]:
results_df = pd.DataFrame(results).sort_values('AUC (test)', ascending=False)
print("\n" + "=" * 80)
print("ITERATION 2: ENRICHED DATA + ENGINEERED FEATURES")
print("=" * 80)
print(results_df.to_string(index=False))


ITERATION 2: ENRICHED DATA + ENGINEERED FEATURES
                   Model  AUC (test)  F1 closed  CV AUC (mean)  CV AUC (std)
       Gradient Boosting      0.6213     0.0108         0.6173        0.0042
                 XGBoost      0.5930     0.0045         0.5947        0.0049
       LogReg (balanced)      0.5715     0.3940         0.5754        0.0053
Random Forest (balanced)      0.5609     0.2293         0.5590        0.0073
               KNN (k=5)      0.5486     0.2313         0.5426        0.0071
Decision Tree (balanced)      0.5241     0.3053         0.5234        0.0073


In [ ]:
for name, pipe in models.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    print(f"\n{'=' * 60}")
    print(f"{name}")
    print(f"{'=' * 60}")
    print(f"AUC: {roc_auc_score(y_test, y_proba):.4f} | F1 closed: {f1_score(y_test, y_pred, pos_label=0):.4f}")
    print(classification_report(y_test, y_pred, target_names=['Closed', 'Success']))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


LogReg (balanced)
AUC: 0.5715 | F1 closed: 0.3940
              precision    recall  f1-score   support

      Closed       0.30      0.57      0.39      2206
     Success       0.79      0.55      0.65      6437

    accuracy                           0.56      8643
   macro avg       0.55      0.56      0.52      8643
weighted avg       0.66      0.56      0.58      8643

Confusion Matrix:
[[1247  959]
 [2877 3560]]

Decision Tree (balanced)
AUC: 0.5241 | F1 closed: 0.3053
              precision    recall  f1-score   support

      Closed       0.29      0.32      0.31      2206
     Success       0.76      0.73      0.74      6437

    accuracy                           0.63      8643
   macro avg       0.52      0.53      0.52      8643
weighted avg       0.64      0.63      0.63      8643

Confusion Matrix:
[[ 711 1495]
 [1740 4697]]

Random Forest (balanced)
AUC: 0.5609 | F1 closed: 0.2293
              precision    recall  f1-score   support

      Closed       0.33      0.17 

In [15]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [16]:
# XGBoost
pipe_xgb = Pipeline([
    ('prep', preprocessor),
    ('model', XGBClassifier(
        random_state=42, eval_metric='logloss',
        scale_pos_weight=(y_train == 1).sum() / (y_train == 0).sum()
    ))
])

print("Tuning XGBoost...")
xgb_grid = RandomizedSearchCV(
    pipe_xgb,
    {
        'model__n_estimators': [100, 200],
        'model__max_depth': [3, 5, 7],
        'model__learning_rate': [0.05, 0.1],
        'model__min_child_weight': [1, 5, 10],
    },
    n_iter=10, cv=5, scoring='roc_auc', n_jobs=-1, random_state=42, verbose=1
)
xgb_grid.fit(X_train, y_train)

y_pred = xgb_grid.predict(X_test)
y_proba = xgb_grid.predict_proba(X_test)[:, 1]
print(f"Best params: {xgb_grid.best_params_}")
print(f"CV AUC: {xgb_grid.best_score_:.4f} | Test AUC: {roc_auc_score(y_test, y_proba):.4f} | F1 closed: {f1_score(y_test, y_pred, pos_label=0):.4f}")

Tuning XGBoost...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best params: {'model__n_estimators': 200, 'model__min_child_weight': 5, 'model__max_depth': 3, 'model__learning_rate': 0.1}
CV AUC: 0.6158 | Test AUC: 0.6185 | F1 closed: 0.0000


In [17]:
# LogReg
pipe_lr = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000, solver='liblinear'))
])

print("Tuning LogReg...")
lr_grid = GridSearchCV(
    pipe_lr,
    {'model__C': [0.01, 0.1, 1, 10], 'model__penalty': ['l1', 'l2']},
    cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
)
lr_grid.fit(X_train, y_train)

y_pred = lr_grid.predict(X_test)
y_proba = lr_grid.predict_proba(X_test)[:, 1]
print(f"Best params: {lr_grid.best_params_}")
print(f"CV AUC: {lr_grid.best_score_:.4f} | Test AUC: {roc_auc_score(y_test, y_proba):.4f} | F1 closed: {f1_score(y_test, y_pred, pos_label=0):.4f}")

Tuning LogReg...
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params: {'model__C': 0.1, 'model__penalty': 'l1'}
CV AUC: 0.6101 | Test AUC: 0.6117 | F1 closed: 0.4161


In [18]:
# Decision Tree
pipe_dt = Pipeline([
    ('prep', preprocessor),
    ('model', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])

print("Tuning Decision Tree...")
dt_grid = GridSearchCV(
    pipe_dt,
    {
        'model__max_depth': [3, 5, 7, 10, 15, None],
        'model__min_samples_leaf': [5, 10, 20, 50],
        'model__min_samples_split': [2, 5, 10],
    },
    cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
)
dt_grid.fit(X_train, y_train)

y_pred = dt_grid.predict(X_test)
y_proba = dt_grid.predict_proba(X_test)[:, 1]
print(f"Best params: {dt_grid.best_params_}")
print(f"CV AUC: {dt_grid.best_score_:.4f} | Test AUC: {roc_auc_score(y_test, y_proba):.4f} | F1 closed: {f1_score(y_test, y_pred, pos_label=0):.4f}")

Tuning Decision Tree...
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best params: {'model__max_depth': 7, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2}
CV AUC: 0.5964 | Test AUC: 0.6024 | F1 closed: 0.4085


In [20]:
numeric_cols = X_train.select_dtypes(include='number')
correlations = numeric_cols.join(y_train).corr()['target'].drop('target').abs().sort_values(ascending=False)
print(correlations)

age_first_funding_days      0.105927
funding_span_days           0.073877
funding_total_usd           0.065598
has_multiple_rounds         0.061168
venture                     0.058733
funding_rounds              0.057494
avg_raised_per_round        0.057149
avg_years_between_rounds    0.047145
round_B                     0.039420
round_A                     0.037284
private_equity              0.033905
round_C                     0.030747
round_D                     0.028174
debt_financing              0.027189
grant                       0.014042
angel                       0.006382
seed                        0.004365
round_E                          NaN
Name: target, dtype: float64


In [24]:
from sklearn.inspection import permutation_importance

# Run on your best model (e.g. LogReg balanced)
perm = permutation_importance(lr_grid.best_estimator_, X_test, y_test,
                              scoring='roc_auc', n_repeats=10,
                              random_state=42, n_jobs=-1)

# Get feature names from the preprocessor
feature_names = lr_grid.best_estimator_.named_steps['prep'].get_feature_names_out()

# Sort and display
importance_df = pd.DataFrame({
    'feature': X_test.columns.tolist(),
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

print(importance_df.to_string())

                     feature  importance_mean  importance_std
1     age_first_funding_days         0.047922        0.005014
19            industry_group         0.022011        0.002289
3          funding_span_days         0.018098        0.004705
0       avg_raised_per_round         0.008008        0.001995
18              region_group         0.005102        0.000747
2        has_multiple_rounds         0.002851        0.001962
13                   round_A         0.001515        0.000715
7                       seed         0.001198        0.000456
12            private_equity         0.001076        0.000752
4   avg_years_between_rounds         0.001016        0.000432
8                    venture         0.000820        0.000413
10                     grant         0.000577        0.000579
6          funding_total_usd         0.000566        0.001092
14                   round_B         0.000352        0.000198
16                   round_D         0.000115        0.000158
11      

In [22]:
print(lr_grid.best_estimator_.named_steps.keys())

dict_keys(['prep', 'model'])
